# Rizzler V24 — Context Convergence & Multi-Agent Live Qualification

**Production acceptance exercise**, not a certificate of production readiness. Run on Kaggle with Internet and a paid Gemini API key added as a Kaggle Secret named `GEMINI_API_KEY`. A GPU T4 may be enabled, but Gemini inference runs through the hosted API. Do not expose tokens in cell output.

This notebook records the actual Rizzler V24 PR56 source SHA at execution time (or uses an explicit `RIZZLER_V24_QUALIFY_SHA` override), validates local contracts, performs real model preflight, creates an actual multi-format SQL/Excel/Word/PPT/PDF job, and separately attempts a real source-research workflow. It saves diagnostics and rendered artifacts only on completion. If anything fails, the notebook **reports the failure** and preserves evidence rather than inventing a successful result.

Run cells in order. Internet access, Gemini model compatibility, billing and rate limits determine whether live qualification can finish.

In [ ]:
# 1. Run isolation and securely load Kaggle / Google Colab secret.
import os, sys, json, subprocess, pathlib, time, secrets, hashlib, datetime, shutil
BASE=pathlib.Path(os.environ.get("RIZZLER_QUALIFICATION_ROOT") or (
    "/kaggle/working/rizzler_v24_qualification"
    if pathlib.Path("/kaggle/working").is_dir()
    else "/content/rizzler_v24_qualification"
    if pathlib.Path("/content").is_dir()
    else str(pathlib.Path.cwd()/"rizzler_v24_qualification"))).resolve()
BASE.mkdir(parents=True,exist_ok=True)
SRC=BASE/"Voice-AI"
BRANCH="feature/v24-context-convergence-and-safe-repair"
PIN=os.environ.get("RIZZLER_V24_QUALIFY_SHA")  # Optional exact revision, not stale hardcoded code.
api_key=os.environ.get("GEMINI_API_KEY","").strip()
if not api_key:
    try:
        from kaggle_secrets import UserSecretsClient
        api_key=UserSecretsClient().get_secret("GEMINI_API_KEY") or ""
    except (ImportError,RuntimeError,OSError,KeyError):
        pass
if not api_key:
    try:
        from google.colab import userdata
        api_key=userdata.get("GEMINI_API_KEY") or ""
    except (ImportError,RuntimeError,OSError,KeyError):
        pass
if not api_key:
    raise RuntimeError("Set GEMINI_API_KEY as a Kaggle/Colab secret before live testing")
os.environ["GEMINI_API_KEY"]=api_key
os.environ["RIZZLER_ACCESS_KEY"]=secrets.token_urlsafe(32)
os.environ["RIZZLER_WORK_ROOT"]=str(BASE/"work")
os.environ["RIZZLER_V23_MODE"]="full"
os.environ["RIZZLER_CONTEXT_CONVERGENCE"]="1"
os.environ["RIZZLER_GEMINI_NATIVE_TOOLS"]="1"
os.environ["RIZZLER_GEMINI_ENABLE_THINKING"]="true"
os.environ["RIZZLER_ARTIFACT_QA"]="strict"
os.environ["RIZZLER_PROVIDER_SLOT_ROOT"]=str(BASE/"rate_governor")
print("Run directory:",BASE)
print("Gemini key: present (never printed)")

In [ ]:
# 2. Clone only the approved branch and record the exact tested revision.
if not SRC.exists():
    subprocess.run(["git","clone","--depth","1","--branch",BRANCH,
                    "https://github.com/SAUNAK359/Voice-AI.git",str(SRC)],check=True)
else:
    subprocess.run(["git","-C",str(SRC),"fetch","--depth","1","origin",BRANCH],check=True)
    subprocess.run(["git","-C",str(SRC),"checkout","--force","FETCH_HEAD"],check=True)
if PIN:
    subprocess.run(["git","-C",str(SRC),"fetch","--depth","1","origin",PIN],check=True)
    subprocess.run(["git","-C",str(SRC),"checkout","--force","FETCH_HEAD"],check=True)
rev=subprocess.check_output(["git","-C",str(SRC),"rev-parse","HEAD"],text=True).strip()
if PIN and rev != PIN:
    raise RuntimeError("Pinned source revision mismatch")
subprocess.run([sys.executable,"-m","pip","install","-q","-e",str(SRC)+"[work,test]",
                "pymupdf>=1.25,<2"],check=True)
if shutil.which("libreoffice") is None:
    subprocess.run(["apt-get","update","-qq"],check=True)
    subprocess.run(["apt-get","install","-y","-qq","libreoffice-writer","libreoffice-impress"],check=True)
if not shutil.which("libreoffice"):
    raise RuntimeError("Strict PPT/Word/PDF QA requires a working LibreOffice executable")
sys.path.insert(0,str(SRC/"src"))
print("Exact source tested:",rev)

In [ ]:
# 3. Minimal offline contracts, isolated from paid Gemini
env=os.environ.copy()
env["RIZZLER_V23_MODE"]="full"
tests=subprocess.run([
    sys.executable,"-m","pytest","-q",
    str(SRC/"tests/test_v24_context_convergence.py"),
    str(SRC/"tests/test_v24_intent_and_recovery.py"),
    str(SRC/"tests/test_v23_flags.py")],
    cwd=str(SRC),env=env,text=True,capture_output=True)
(BASE/"v24_contract_tests.txt").write_text(tests.stdout+"\n"+tests.stderr)
print(tests.stdout[-3000:])
assert tests.returncode==0, "Local V24 contract suite failed; examine v24_contract_tests.txt"

In [ ]:
# 4. Real Gemini compatibility/billing preflight
from google import genai
from google.genai import types
from rizzler.config import Settings
from rizzler.v23_flags import enabled
settings=Settings.from_env()
assert enabled("RIZZLER_CONTEXT_CONVERGENCE") and enabled("RIZZLER_GEMINI_NATIVE_TOOLS")
assert settings.gemini_enable_thinking, "V24 default thinking unexpectedly disabled"
client=genai.Client(api_key=api_key)
model=settings.gemini_model
try:
    model_info=client.models.get(model=model)
    print("Gemini model lookup succeeded:",model)
    response=client.models.generate_content(
        model=model,contents="Reply with the exact text: V24_API_OK",
        config=types.GenerateContentConfig(temperature=0,max_output_tokens=40))
    assert "V24_API_OK" in (response.text or ""), "Gemini model text validation failed"
    print("Gemini inference:", response.text.strip())
finally:
    client.close()

## Live multi-format job: synthetic audited data

The next job uses an explicitly **synthetic** three-year trade dataset with exact arithmetic. Do not treat these values as actual Indian trade statistics. This verifies source SQL, multi-agent cooperation, PowerPoint/Word/Excel/PDF generation and final artifact checks without mixing fabricated data into external research.

In [ ]:
# 5. Create audited synthetic CSV and register it in an isolated WorkStore
import csv, io
from rizzler.work_store import WorkStore
from rizzler.work_worker import WorkWorker
store=WorkStore(BASE/"work")
owner="kaggle-v24-qualification"
table=[
    {"year":"2023","exports_usd_mn":80,"imports_usd_mn":110},
    {"year":"2024","exports_usd_mn":95,"imports_usd_mn":115},
    {"year":"2025","exports_usd_mn":125,"imports_usd_mn":150},
]
csv_io=io.StringIO()
writer=csv.DictWriter(csv_io,fieldnames=list(table[0]))
writer.writeheader()
writer.writerows(table)
uploaded=store.register_upload(owner,"synthetic_trade.csv",csv_io.getvalue().encode())
print("Synthetic data:",table)
print("Dataset upload ID:",uploaded["id"])

In [ ]:
# 6. Submit the actual durable Work Agent multi-artifact task
formats=["data_analysis","pptx","docx","xlsx","pdf"]
objective=(
 "Analyze only the attached clearly labeled SYNTHETIC dataset, with exact "
 "imports, exports, trade balance (exports minus imports), year over year "
 "changes and limitations. Produce a professional editable presentation, "
 "Word report, Excel workbook with checked formulas where appropriate and "
 "a PDF report. Use USD millions exactly as present in the dataset; no "
 "unstated external sources or invented current economic figures. "
 "Verify cross-format consistency, evidence dependencies and output quality."
)
job=store.submit(owner,objective,formats,
                 "v24-synthetic-multiformat-001",[uploaded["id"]])
print("Created multi-format job",job["id"],job["status"])

In [ ]:
# 7. Run real worker from an existing Jupyter loop; report actual job status.
start=time.monotonic()
worker_error=None
try:
    handled=await WorkWorker(store).once()
except Exception as exc:
    worker_error=type(exc).__name__
    handled=False
elapsed=round(time.monotonic()-start,2)
multi=store.get(owner,job["id"])
print("Worker processed a claim:",handled,"elapsed_s:",elapsed)
print("Actual job result:",multi["status"],multi["stage"],multi["error_code"])
steps=store.agent_steps(owner,job["id"])
print("Agents:",[(a["agent"],a["status"],a["error_code"]) for a in steps])
report={"job":multi,"seconds":elapsed,"worker_error":worker_error,
        "agents":steps,"questions":store.questions(owner,job["id"]),
        "events":store.events(owner,job["id"]),
        "diagnostics":store.diagnostics(owner,job["id"])}
(BASE/"synthetic_status.json").write_text(json.dumps(report,indent=2,default=str))
print("Multi-format qualification:", "PASS" if multi["status"]=="completed" else "NOT_QUALIFIED")

In [ ]:
# 8. Inspect actually published artifacts. Never stop before exporting evidence.
from pptx import Presentation
from docx import Document
from openpyxl import load_workbook
import fitz
all_artifacts=store.artifacts(owner,job["id"])
required={"pptx","docx","xlsx","pdf"}
validation=[]
errors=[]
if multi["status"]=="completed":
    for row in all_artifacts:
        try:
            path,record=store.artifact(owner,row["id"])
            data=path.read_bytes()
            if hashlib.sha256(data).hexdigest()!=row["sha256"]:
                raise ValueError("Artifact checksum mismatch")
            if row["kind"]=="pptx":
                assert len(Presentation(str(path)).slides)>=3
            elif row["kind"]=="docx":
                assert len(Document(str(path)).paragraphs)>=3
            elif row["kind"]=="xlsx":
                assert len(load_workbook(str(path),data_only=False).worksheets)>=1
            elif row["kind"]=="pdf":
                with fitz.open(str(path)) as reader:
                    assert len(reader)>=1
            validation.append({"kind":row["kind"],"sha256":row["sha256"]})
        except Exception as exc:
            errors.append({"kind":row["kind"],"error_type":type(exc).__name__})
else:
    errors.append({"reason":"job_not_completed","status":multi["status"]})
artifact_gate={
    "required":sorted(required),
    "observed":sorted(x["kind"] for x in all_artifacts),
    "missing":sorted(required-{x["kind"] for x in all_artifacts}),
    "verified":validation,"problems":errors,
}
artifact_gate["passed"]=multi["status"]=="completed" and not artifact_gate["missing"] and not errors
(BASE/"synthetic_artifact_gate.json").write_text(json.dumps(artifact_gate,indent=2))
print("Artifact verification:", "PASS" if artifact_gate["passed"] else "NOT_QUALIFIED")

## Separate real grounded research workflow

This is not a mock: the model must perform real Google Search and the existing Rizzler verified-quote/source-retrieval gates. Unavailable or inaccessible authoritative sources are reported, not treated as successful evidence. Output review and cross-format consistency must be inspected before a production claim.

In [ ]:
# 9. Submit an independent real research job
research_objective=(
 "Investigate India's merchandise imports and exports for fiscal year "
 "2024-25 using official public sources. Compare total exports versus total "
 "imports and their balance, explicitly distinguish goods and services, "
 "period definitions and currency units. Cite the source URLs you actually "
 "retrieved; identify uncertainty and gaps. Prepare a sourced PPTX, Word "
 "report and PDF for an analytical stakeholder. No fabricated figures."
)
research_job=store.submit(owner,research_objective,
                           ["research","pptx","docx","pdf"],
                           "v24-research-live-001",[])
print("Research job:",research_job["id"],research_job["status"])

In [ ]:
# 10. Execute grounded research without a nested asyncio.run.
t0=time.monotonic()
research_worker_error=None
try:
    claimed=await WorkWorker(store).once()
except Exception as exc:
    claimed=False
    research_worker_error=type(exc).__name__
research_status=store.get(owner,research_job["id"])
print("Worker processed claim:",claimed,"elapsed_s:",round(time.monotonic()-t0,2))
print("Research result:",research_status["status"],research_status["error_code"])
steps_research=store.agent_steps(owner,research_job["id"])
print("Agent stages:",[(a["agent"],a["status"],a["error_code"]) for a in steps_research])
research_evidence={
    "job":research_status,"worker_error":research_worker_error,
    "agents":steps_research,"questions":store.questions(owner,research_job["id"]),
    "events":store.events(owner,research_job["id"]),
    "diagnostics":store.diagnostics(owner,research_job["id"]),
    "cognitive_events":store.cognitive_events(owner,research_job["id"],limit=200)
}
(BASE/"research_status.json").write_text(json.dumps(research_evidence,indent=2,default=str))
print("Research qualification:", "PASS" if research_status["status"]=="completed" else "NOT_QUALIFIED")

In [ ]:
# 11. Always export evidence, including failed job stages; never export secrets.
import zipfile
pack=BASE/"Rizzler_V24_Live_Qualification_Evidence.zip"
manifest={"source_sha":rev,"model_id":model,
          "mode":"full","thinking":settings.gemini_enable_thinking,
          "synthetic_job_status":multi["status"],
          "research_job_status":research_status["status"],
          "synthetic_artifact_passed":artifact_gate["passed"],
          "created_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),
          "limitations":"No API keys or private databases; content, factual, visual and deployment QA remain necessary"}
with zipfile.ZipFile(pack,"w",zipfile.ZIP_DEFLATED) as z:
    for name in ("v24_contract_tests.txt","synthetic_status.json",
                 "synthetic_artifact_gate.json","research_status.json"):
        file=BASE/name
        if file.is_file():
            z.write(file,arcname=name)
    z.writestr("MANIFEST.json",json.dumps(manifest,indent=2))
print("Evidence bundle:",pack,"bytes:",pack.stat().st_size)
print("Release status:", "NEEDS_REVIEW" if
      artifact_gate["passed"] and research_status["status"]=="completed"
      else "NOT_QUALIFIED")

### Release conclusion

Even if all cells pass, perform human/independent visual review of the generated PPTX/DOCX/PDF, verify Excel formulas against input data, trace every factual research claim to the retrieved primary source, repeat under provider timeouts and test job recovery across process restarts. **Green tests are not an automatic certification.** Download the Evidence ZIP from Kaggle Output and compare its manifest and diagnostics with the PR56 source.